# xGEMS tutorial 3b — basalt glass dissolution, **open to atmospheric CO₂**


**Authors: G. Dan Miron**

**Formatted with Claude Opus**


Same kinetics as tutorial 3 (10 g N-MORB glass in 1 kg water, 25 °C, 1 bar, Gíslason &
Oelkers 2003 rate law, far-from-equilibrium so $1-Q/K\approx1$) — but now the fluid is
**open to the atmosphere**: the dissolved CO₂ is held at its Henry's-law equilibrium with a
fixed atmospheric $f_{\mathrm{CO}_2}$ at every step.

**What "fixed fugacity" means here.** Molecular CO₂(aq) is pinned to
$[\mathrm{CO_2(aq)}] = K_H\,f_{\mathrm{CO}_2}$ (Henry's law). As the glass dissolves and pH
rises, carbon speciates into HCO₃⁻/CO₃²⁻, so the *total* dissolved carbon must grow to keep
CO₂(aq) fixed — i.e. the water keeps absorbing CO₂ from the atmosphere. At each step we
root-find the carbon to add so GEMS returns the target CO₂(aq).

The starting water is the Table-1 ion composition **without its carbon** (carbon is now set
by the atmosphere) in 1 kg of water.

> Runs against your local `xgems` env + `gems_files/SW-B_titr3-dat.lst`; not executed here.


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from xgems import ChemicalEngineDicts, Material

gems = ChemicalEngineDicts("gems_files/SW-B_titr3-dat.lst")

T = 25.0 + 273.15      # K   (25 °C)
P = 1.0e5              # Pa  (1 bar)
YEAR = 365.25 * 24 * 3600.0

## 2. Materials — starting water (no carbon) and N-MORB glass

Carbon is left out of the water: it will be supplied by the atmosphere through the CO₂
buffer. Everything else is 1 kg of water + the Table-1 ions.

In [ ]:
# Vellankatla ions in 1 kg water, WITHOUT carbon (CO2 comes from the atmosphere)
water = Material(gems, "vellankatla_open")
water.add("H2O", 1.0, "kg")            # 1 kg of water
water.add({
    "Si": 256e-6, "Na": 269e-6, "K": 11.9e-6, "Ca": 71e-6, "Mg": 38e-6,
    "Fe": 0.16e-6, "Al": 1.09e-6, "S": 15e-6, "Cl": 120e-6,
})

# N-MORB basaltic glass (Table 2), oxide wt% as grams; Fe as total FeO
morb = Material(gems, "N_MORB_glass")
morb.add("SiO2",  50.45, "g"); morb.add("Al2O3", 15.25, "g"); morb.add("FeO", 10.43, "g")
morb.add("CaO",   11.30, "g"); morb.add("MgO",    7.58, "g"); morb.add("Na2O", 2.68, "g")
morb.add("TiO2",   1.62, "g"); morb.add("K2O",    0.09, "g")

n_Si_per_g = morb.b_dict()["Si"] / (morb.mass_kg() * 1000.0)   # mol Si per gram of glass

## 3. Rate law and the atmospheric CO₂ buffer

The rate law is unchanged. The buffer adds carbon until the CO₂(aq) **activity** matches its atmospheric equilibrium; with $f_{\mathrm{CO}_2}=10^{-3.5}$ bar and $K_H(25^\circ\mathrm C)=10^{-1.47}$
mol kg⁻¹ bar⁻¹ that target is $[\mathrm{CO_2(aq)}]=10^{-4.97}\approx1.1\times10^{-5}$
(activity, ≈ molality in dilute water).

In [ ]:
K_GEO = 10**-5.6; E_A = 25.5e3; R_GAS = 8.314; A_GEO = 250.0

def glass_rate(area_cm2, a_H, a_Al):
    arrhenius = np.exp(-E_A / (R_GAS * T))
    return area_cm2 * K_GEO * arrhenius * (a_H**3 / a_Al)**(1.0/3.0)   # (1-Q/K)=1

# --- atmospheric CO2 buffer ------------------------------------------------
PCO2_LOG = -3.5        # log10 atmospheric fCO2 (bar); modern air is closer to -3.4
KH_LOG   = -1.47       # log10 Henry constant for CO2 at 25 C (mol/kg/bar)
CO2AQ_TARGET = 10 ** (KH_LOG + PCO2_LOG)      # CO2(aq) ACTIVITY to hold (Henry's law)

def co2aq_activity(engine):
    # activity of molecular CO2(aq); in dilute water this ~ its molality, but the
    # activity is the correct quantity for Henry's law and stays valid in brines.
    ln_a = engine.species_ln_activities
    for nm in ("CO2@", "CO2(aq)", "CO2", "CO2*"):
        if nm in ln_a:
            return np.exp(ln_a[nm])
    raise KeyError("aqueous CO2 species not found — set its exact name for your database")

def equilibrate_open_CO2(base_mix):
    # add CO2 until the CO2(aq) activity equals the atmospheric-equilibrium value
    def resid(logC):
        c_add = Material(gems, "co2").add({"C": 1, "O": 2}, 10**logC, "mol")
        gems.equilibrate(T, P, base_mix + c_add)
        return np.log(co2aq_activity(gems)) - np.log(CO2AQ_TARGET)
    logC = brentq(resid, -8.0, 0.5, xtol=1e-4, maxiter=60)
    resid(logC)                     # leave the engine at the solved state
    return 10 ** logC               # total dissolved carbon (mol)

## 4. Kinetic loop (CO₂-buffered)

Identical to tutorial 3, except each step is equilibrated **through the CO₂ buffer** instead
of at fixed total carbon. This runs many equilibrations per step (the root-find), so it is
slower — reduce the number of time points if needed.

In [ ]:
m0 = 10.0
A0 = A_GEO * m0
times = np.logspace(-3, 1, 45) * YEAR      # 0.001 -> 10 years

def activity(engine, *names, default):
    ln_a = engine.species_ln_activities
    for nm in names:
        if nm in ln_a:
            return np.exp(ln_a[nm])
    return default

m = m0
t_prev = 0.0
rows = []
for t in times:
    dt = t - t_prev
    g_dissolved = m0 - m

    base = water if g_dissolved <= 1e-12 else water + morb.set_quantity(g_dissolved, "g")
    C_tot = equilibrate_open_CO2(base)          # buffered equilibrium at atm fCO2

    a_H  = 10.0 ** (-gems.pH)
    a_Al = activity(gems, "Al+3", "Al3+", default=1e-30)
    area = A0 * (m / m0) ** (2.0/3.0)
    rate = glass_rate(area, a_H, a_Al)

    row = {"years": t / YEAR, "g_dissolved": g_dissolved, "pH": gems.pH, "C_total": C_tot}
    for el in ["Si", "C", "Al", "Ca", "Mg", "Fe", "Na", "K"]:
        row["aq_" + el] = gems.aq_elements_molality.get(el, 0.0)
    row.update(gems.phases_moles)
    rows.append(row)

    m = max(m - rate / n_Si_per_g * dt, 0.0)
    t_prev = t

    print(f"t = {t/YEAR:.4g} yr   pH = {gems.pH:.2f}   dissolved = {g_dissolved:.3f} g")

hist = pd.DataFrame(rows).set_index("years")
print("dissolved by 10 yr:", round(hist['g_dissolved'].iloc[-1], 3), "g;  final pH",
      round(hist['pH'].iloc[-1], 2))
hist[["pH", "g_dissolved", "C_total", "aq_Si"]].iloc[::8]

## 5. pH evolution

With CO₂ replenished from the atmosphere, pH is buffered lower than the closed case (compare the open vs closed curves in the paper's Fig. 1).

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.semilogx(hist.index, hist["pH"], color="k")
plt.xlabel("ξ-time (years)"); plt.ylabel("pH")
plt.title("Fluid pH — open to atmospheric CO₂ (25 °C, 1 bar)")
plt.grid(True, which="both", ls=":", lw=0.5)
plt.tight_layout(); plt.show()

## 6. Secondary minerals formed

In [ ]:
skip = ("aq", "gas", "fluid")
mineral_cols = [p for p in gems.phase_names
                if not p.lower().startswith(skip) and p in hist.columns
                and hist[p].max() > 1e-9]

plt.figure(figsize=(7.5, 5))
for p in mineral_cols:
    plt.loglog(hist.index, hist[p].clip(lower=1e-12), label=p)
plt.xlabel("ξ-time (years)"); plt.ylabel("Secondary minerals formed (moles)")
plt.ylim(1e-7, 1e-1)
plt.title("Secondary mineralogy — open to atmospheric CO₂")
plt.legend(fontsize=8, ncol=2); plt.grid(True, which="both", ls=":", lw=0.5)
plt.tight_layout(); plt.show()

## 7. Aqueous composition

Now the **CO₂** curve (total dissolved carbon) rises steadily, because the water keeps taking up atmospheric CO₂ as it neutralises — the open-system signature.

In [ ]:
labels = {"Si": "Si", "C": "CO2", "Na": "Na", "Ca": "Ca",
          "K": "K", "Mg": "Mg", "Al": "Al", "Fe": "Fe"}

plt.figure(figsize=(7.5, 5.5))
for el, name in labels.items():
    y = hist["aq_" + el].clip(lower=1e-12) * 1000.0     # mol/kg -> mmol/kg
    plt.plot(hist.index, y)
    plt.text(hist.index[-1] * 1.15, y.iloc[-1], name, fontsize=9, va="center")

plt.xscale("log"); plt.yscale("log")
plt.xlim(1e-3, 3e1); plt.ylim(1e-6, 2e1)
plt.xlabel("ξ-time (years)"); plt.ylabel("Dissolved elements (mmol/kg)")
plt.title("Solution composition — open to atmospheric CO₂")
plt.grid(True, which="both", ls=":", lw=0.4)
plt.tight_layout(); plt.show()

---
**Notes.** `PCO2_LOG` sets the atmospheric fugacity — change it for palaeo or elevated-CO₂
scenarios. The buffer targets the dissolved **CO₂(aq)** species, so its name
(`"CO2@"`, `"CO2(aq)"`, …) must match your database; the helper tries the common ones. The
root-find makes this notebook noticeably slower than the closed version — lower the number
of time points while prototyping.